# Saluki for synonymous Gluc CDS design

[Notebook](saluki-gluc-cds.ipynb) · [Installation](../docs/installation.md)

Optimize the Gluc CDS for predicted mRNA stability. Keep translation and both UTRs fixed.

## 1. Load the model



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

from pathlib import Path

import torch
from bioseq_models import load_model
from MotifSift import call_motifs, generate_designs, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
cds = (
    "ATGGGAGTCAAAGTTCTGTTTGCCCTGATCTGCATCGCTGTGGCCGAGGCCAAGCCCACCGAGAACAACGAA"
    "GACTTCAACATCGTGGCCGTGGCCAGCAACTTCGCGACCACGGATCTCGATGCTGACCGCGGGAAGTTGCCC"
    "GGCAAGAAGCTGCCGCTGGAGGTGCTCAAAGAGATGGAAGCCAATGCCCGGAAAGCTGGCTGCACCAGGGGC"
    "TGTCTGATCTGCCTGTCCCACATCAAGTGCACGCCCAAGATGAAGAAGTTCATCCCAGGACGCTGCCACACC"
    "TACGAAGGCGACAAAGAGTCCGCACAGGGCGGCATAGGCGAGGCGATCGTCGACATTCCTGAGATTCCTGGG"
    "TTCAAGGACTTGGAGCCCATGGAGCAGTTCATCGCACAGGTCGATCTGTGTGTGGACTGCACAACTGGCTGC"
    "CTCAAAGGGCTTGCCAACGTGCAGTGTTCTGACCTGCTCAAGAAGTGGCTGCCGCAACGCTGTGCGACCTTT"
    "GCCAGCAAGATCCAGGGCCAGGTGGACAAGATCAAGGGGGCCGGTGGTGACTAA"
)
utr5 = "ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTCAAACAGACACC"
utr3 = "GCTGGAGCCTCGGTGGCCTAGCTTCTTGCCCCTTGGGCCTCCCCCCAGCCCCTCCTCCCCTTCCTGCACCCGTACCCCCGTGGTCTTTGAATAAAGTCTGAGTGGGCGGCA"
saluki = load_model(
    "saluki", device=device,
    cds_start=len(utr5), cds_end=len(utr5) + len(cds),
    exon_lengths=[len(utr5) + len(cds) + len(utr3)],
)


def encode_utr(sequence):
    indices = torch.tensor(["ACGT".index(base) for base in sequence])
    return torch.nn.functional.one_hot(indices, num_classes=4)


left, right = encode_utr(utr5), encode_utr(utr3)


class CDSScorer:
    task_names = saluki.task_names
    device = saluki.device

    def score(self, sequences):
        return saluki.score([utr5 + sequence + utr3 for sequence in sequences])

    def score_soft(self, probabilities):
        batch = len(probabilities)
        full_transcripts = torch.cat([
            left.to(probabilities).expand(batch, -1, -1),
            probabilities,
            right.to(probabilities).expand(batch, -1, -1),
        ], dim=1)
        return saluki.score_soft(full_transcripts.clamp(0, 1))


model = CDSScorer()
seeds = [cds]

run = Path("outputs/saluki_gluc_cds")




## 2. Generate candidates



In [2]:
designs = generate_designs(
    model, length=len(cds), initial=seeds, objective="maximize",
    gain_fractions=(0.5, 0.9), output_dir=run / "designs",
    is_CDS_optimized=True, fixed_region=[slice(len(cds) - 3, len(cds))],
)
designs.groupby(["algorithm", "gain_fraction"])["score"].agg(["size", "median", "max"])


size    median       max
algorithm                       gain_fraction                          
adalead                         0.5             100  1.354763  1.582023
                                0.9             100  2.352451  2.409209
basesearch                      0.5             100  1.379188  1.581755
                                0.9             100  2.365475  2.432327
beam_search                     0.5             100  1.341735  1.753246
                                0.9             100  2.387578  2.440444
cross_entropy_method            0.5             100  1.395321  1.621385
                                0.9             100  2.362609  2.422931
den                             0.5             100  1.250732  1.634838
                                0.9             100  2.070580  2.194720
fast_seqprop                    0.5             100  1.281025  1.714201
                                0.9             100  2.040004  2.166345
genetic_algorithm               0.5             100  1.378127  1.598297
                                0.9             100  2.351374  2.422093
ism_greedy                      0.5             100  1.322589  1.479062
                                0.9             100  2.214530  2.328416
ledidi                          0.5             100  0.771945  1.050244
                                0.9             100  1.246182  1.477367
simulated_annealing             0.5             100  0.496737  0.912555
                                0.9             100  0.719633  1.034878
vanilla_activation_maximization 0.5             100  1.441738  1.705358
                                0.9             100  2.078668  2.224066



## 3. Discover motifs

Use the 90% gain stage for motif discovery and panel selection.



In [3]:
designs = designs.loc[designs["gain_fraction"].eq(0.9)].copy()
motifs = call_motifs(designs, model, reverse_complement=False, output_dir=run / "motifs")
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-GUKGAUU,640,542,2377.0
1,2-CCACUGG,329,329,997.0
4,5-UCGCGAG,120,118,545.0
5,6-AAUGCAG,85,85,367.0
7,8-AYCUGCUGAA,107,107,779.0
8,10-AGUUGCCAGG,20,20,381.0
9,9-AGCAGUUUA,31,31,1219.0
10,11-GAUGCCUUAA,277,253,961.0
12,14-UCGMUCWR,25,25,1007.0
15,17-UUGCGCAG,10,10,217.0




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,ATGGGGGTCAAGGTTTTGTTCGCGCTAATTTGCATCGCGGTTGCGG...,2.440444,1-GUKGAUU;2-CCACUGG;5-UCGCGAG;8-AYCUGCUGAA;10-...
773,2,ATGGGCGTTAAGGTCTTGTTCGCGTTGATCTGTATCGCGGTTGCGG...,2.056324,1-GUKGAUU;11-GAUGCCUUAA;14-UCGMUCWR;21-UCGCGCU...
660,3,ATGGGCGTTAAGGTCTTGTTCGCGCTCATCTGCATCGCGGTCGCGG...,2.105933,1-GUKGAUU;8-AYCUGCUGAA;11-GAUGCCUUAA;21-UCGCGC...
768,4,ATGGGCGTTAAGGTCTTGTTCGCGCTCATCTGCATCGCGGTTGCGG...,2.058027,1-GUKGAUU;5-UCGCGAG;9-AGCAGUUUA;14-UCGMUCWR;21...
870,5,ATGGGCGTTAAGGTCTTGTTCGCGCTCATATGCATCGCGGTGGCGG...,2.028798,1-GUKGAUU;10-AGUUGCCAGG;9-AGCAGUUUA;11-GAUGCCU...
767,6,ATGGGGGTTAAGGTCTTGTTTGCGTTGATCTGCATCGCGGTGGCCG...,2.058215,1-GUKGAUU;2-CCACUGG;9-AGCAGUUUA;11-GAUGCCUUAA;...
150,7,ATGGGCGTGAAGGTTTTGTTCGCGCTCATTTGCATCGCGGTGGCGG...,2.378805,1-GUKGAUU;2-CCACUGG;6-AAUGCAG;9-AGCAGUUUA;11-G...
748,8,ATGGGCGTCAAGGTTCTCTTCGCGCTCATCTGCATCGCGGTCGCCG...,2.064030,1-GUKGAUU;2-CCACUGG;6-AAUGCAG;8-AYCUGCUGAA;14-...
474,9,ATGGGGGTTAAGGTTTTGTTCGCGCTTATCTGCATCGCGGTGGCTG...,2.335562,1-GUKGAUU;2-CCACUGG;5-UCGCGAG;9-AGCAGUUUA;11-G...
764,10,ATGGGGGTGAAGGTCCTGTTCGCGCTTATCTGTATCGCGGTGGCGG...,2.058561,1-GUKGAUU;6-AAUGCAG;8-AYCUGCUGAA;10-AGUUGCCAGG...
